In [7]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import random
import matplotlib.pyplot as plt

# Set global seed for reproducibility
np.random.seed(0)
random.seed(0)

# Load and preprocess data
def load_CSN_data():
    csv_path = '../CSN_Viability_Database_2025.csv'
    return pd.read_csv(csv_path)

CSN = load_CSN_data()

# Drop unused columns
CSN = CSN.drop(['Example ID', 'Source', 'Figure ID', 'Data Provider', 'PI',
                'Date Received', 'Data Measurment Published', 'Prior Exposure', 'Comments', 'Error'], axis=1)

# One-hot encode categorical variables
CSN_prepared = pd.get_dummies(CSN, dtype=int)

# Add engineered features
CSN_prepared['Surface Area per Liter'] = CSN_prepared['Surface Area (NMC) (m2/g)'] * CSN_prepared['Concentration (mg/L)']
CSN_prepared = CSN_prepared.drop(['Surface Area (NMC) (m2/g)'], axis=1)
CSN_prepared['log Concentration'] = np.log10(CSN_prepared['Concentration (mg/L)'] + 1e-9)
CSN_prepared = CSN_prepared.drop(['Concentration (mg/L)'], axis=1)

# Split data
CSN_new = CSN_prepared[-38:]
CSN_prepared = CSN_prepared.drop(CSN_prepared.index[-38:])
X_train = CSN_prepared.drop(['Viability_Fraction'], axis=1)
Y_train = CSN_prepared['Viability_Fraction']
X_test = CSN_new.drop(['Viability_Fraction'], axis=1)
Y_test = CSN_new['Viability_Fraction']

# ----------------------------------------
# Fixed XGBoost parameters (no grid search)
# ----------------------------------------
fixed_params = {
    'n_estimators': 100,
    'max_depth': 7,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
}

# ----------------------------------------
# Run 100 times with different seeds
# ----------------------------------------
mae_list = []
random_seeds = random.sample(range(0, 100000), 100)

for seed in random_seeds:
    model = XGBRegressor(random_state=seed, **fixed_params)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)
    test_mae = mean_absolute_error(Y_test, y_pred)
    mae_list.append(test_mae)
    print(f"Seed {seed}: MAE = {test_mae:.4f}")

# Summary statistics
mae_array = np.array(mae_list)
print("\nSummary of XGBoost predictions over 100 random seeds:")
print("Mean MAE: {:.4f} ± {:.4f}".format(mae_array.mean(), mae_array.std()))
print("Min MAE: {:.4f}".format(mae_array.min()))
print("Max MAE: {:.4f}".format(mae_array.max()))


Seed 50494: MAE = 0.2126
Seed 99346: MAE = 0.3117
Seed 55125: MAE = 0.2531
Seed 5306: MAE = 0.2718
Seed 33936: MAE = 0.2726
Seed 67013: MAE = 0.3211
Seed 63691: MAE = 0.2350
Seed 53075: MAE = 0.2614
Seed 39755: MAE = 0.2822
Seed 62468: MAE = 0.2715
Seed 46930: MAE = 0.3110
Seed 76465: MAE = 0.3037
Seed 28631: MAE = 0.2974
Seed 66150: MAE = 0.3148
Seed 18254: MAE = 0.2806
Seed 36941: MAE = 0.2807
Seed 18316: MAE = 0.2654
Seed 99064: MAE = 0.2520
Seed 12429: MAE = 0.2932
Seed 81050: MAE = 0.2815
Seed 32834: MAE = 0.2578
Seed 69804: MAE = 0.3135
Seed 92428: MAE = 0.3110
Seed 78892: MAE = 0.3052
Seed 19262: MAE = 0.3060
Seed 40651: MAE = 0.3314
Seed 12945: MAE = 0.3260
Seed 95660: MAE = 0.2802
Seed 9665: MAE = 0.3286
Seed 89651: MAE = 0.2939
Seed 43279: MAE = 0.2637
Seed 61884: MAE = 0.3164
Seed 73375: MAE = 0.2996
Seed 13199: MAE = 0.3404
Seed 46372: MAE = 0.3051
Seed 56907: MAE = 0.2932
Seed 41444: MAE = 0.2713
Seed 80070: MAE = 0.2415
Seed 83941: MAE = 0.2589
Seed 26801: MAE = 0.2724
Se